# 🎸 Proyek Data Warehouse: Bandcamp Sales Analytics
Proyek ini bertujuan untuk membangun sebuah **Data Warehouse** berbasis *Star Schema* untuk menganalisis data penjualan musik independen di platform Bandcamp.

### **Alur Kerja (Pipeline):**
1. **Extract & Validate**: Membaca data mentah dan melakukan pembersihan anomali (Data Quality Check).
2. **Transform**: Mengonversi data mentah menjadi tabel Dimensi (Waktu, Lokasi, Artis, Item, Penggemar) dan tabel Fakta.
3. **Load**: Memasukkan data yang telah bertransformasi ke dalam database SQLite.
4. **OLAP Analysis**: Menjalankan 7 query analitik untuk menjawab pertanyaan bisnis dan mengekspor hasilnya untuk visualisasi dashboard.

In [1]:
import pandas as pd
import sqlite3
import hashlib
import numpy as np
import os
import warnings

# Konfigurasi Tampilan & Penyiapan Folder Output
warnings.filterwarnings('ignore')
os.makedirs('database', exist_ok=True)
os.makedirs('etl', exist_ok=True)
os.makedirs('olap', exist_ok=True)

# Path Dataset (Ubah ke 'dataset/1000000-bandcamp-sales.csv' jika ingin memproses full data)
RAW_DATA_PATH = 'dataset/sample_bandcamp_sales.csv' 

print("✅ Library berhasil di-import dan environment telah siap.")

✅ Library berhasil di-import dan environment telah siap.


### **Tahap 1: Extract & Validation**
Pada tahap ini, kita memuat data dari file CSV mentah. Sebelum diproses lebih lanjut, data harus melewati filter **Data Quality Check** untuk memastikan tidak ada data kotor yang masuk ke Data Warehouse.

**Anomali yang ditangani:**
* Menghapus duplikasi transaksi berdasarkan ID Unik (`_id`).
* Menghapus baris jika nilai pada kolom krusial kosong (NULL).
* Menghapus data anomali di mana nominal pendapatan bernilai negatif.
* Menstandardisasi tipe item agar konsisten.

In [2]:
df_raw = pd.read_csv(RAW_DATA_PATH)

def validate_data(df):
    initial_count = len(df)
    
    # 1. Hapus Duplikasi
    df = df.drop_duplicates(subset=['_id'], keep='first')
    
    # 2. Hapus Null pada kolom krusial (Kolom opsional seperti album_title dibiarkan)
    df = df.dropna(subset=['_id', 'artist_name', 'amount_paid_usd', 'item_type'])
    
    # 3. Hapus nilai negatif
    df = df[df['amount_paid_usd'] >= 0]
    
    # 4. Standardisasi item_type
    valid_types = ['a', 't', 'p']
    df.loc[~df['item_type'].isin(valid_types), 'item_type'] = 'other'
    
    print(f"Audit: {initial_count - len(df)} baris anomali berhasil dihapus.")
    print(f"Data Bersih: {len(df)} baris siap diproses.")
    return df

df_clean = validate_data(df_raw)

Audit: 0 baris anomali berhasil dihapus.
Data Bersih: 1000 baris siap diproses.


### **Tahap 2: Transformasi (Pembuatan Tabel Dimensi)**
Data mentah yang sudah bersih kini dipecah menjadi beberapa tabel Dimensi pendukung. Kita menggunakan library `hashlib` (MD5) untuk membuat ID unik secara deterministik untuk data teks seperti nama artis, lokasi, dan item.

In [3]:
# Fungsi Helper untuk membuat ID Unik
def make_id(text, prefix=""):
    if pd.isna(text): return "UNKNOWN"
    return prefix + hashlib.md5(str(text).encode('utf-8')).hexdigest()[:8].upper()

# --- A. Dim_Waktu ---
df_clean['datetime'] = pd.to_datetime(df_clean['utc_date'], unit='s')
df_clean['Time_ID'] = df_clean['datetime'].dt.strftime('%Y%m%d').astype(int)

dim_waktu = df_clean[['Time_ID']].drop_duplicates()
dim_waktu['Tanggal'] = pd.to_datetime(df_clean['datetime']).dt.date
dim_waktu['Bulan'] = pd.to_datetime(df_clean['datetime']).dt.month
dim_waktu['Kuartal'] = pd.to_datetime(df_clean['datetime']).dt.quarter
dim_waktu['Tahun'] = pd.to_datetime(df_clean['datetime']).dt.year

# Logika Bandcamp Friday: Jumat pertama setiap bulan
is_friday = pd.to_datetime(dim_waktu['Tanggal']).dt.dayofweek == 4
is_first_week = pd.to_datetime(dim_waktu['Tanggal']).dt.day <= 7
dim_waktu['Bandcamp_Friday'] = is_friday & is_first_week

# --- B. Dim_Artis ---
df_clean['Artist_ID'] = df_clean['artist_name'].apply(lambda x: make_id(x, "ART-"))
dim_artis = df_clean[['Artist_ID', 'artist_name']].drop_duplicates()
dim_artis.columns = ['Artist_ID', 'Nama_Artis']
dim_artis['Primary_Genre'] = 'Indie / Alternative' # Dummy genre

# --- C. Dim_Lokasi ---
df_clean['Location_ID'] = df_clean['country_code'].apply(lambda x: make_id(x, "LOC-"))
dim_lokasi = df_clean[['Location_ID', 'country', 'country_code']].drop_duplicates()
dim_lokasi.columns = ['Location_ID', 'Negara', 'Kode_Negara']

# --- D. Dim_Item ---
df_clean['Item_ID'] = df_clean['item_description'].apply(lambda x: make_id(x, "ITM-"))
type_mapping = {'a': 'Digital Album', 't': 'Digital Track', 'p': 'Physical / Merch', 'other': 'Other'}
df_clean['Tipe_Item'] = df_clean['item_type'].map(type_mapping)

dim_item = df_clean[['Item_ID', 'item_description', 'Tipe_Item', 'item_price']].drop_duplicates()
dim_item.columns = ['Item_ID', 'Nama_Item', 'Tipe_Item', 'Harga_Satuan']

# --- E. Dim_Penggemar ---
np.random.seed(42)
df_clean['User_ID'] = "USR-" + np.random.randint(1, 500, df_clean.shape[0]).astype(str)
dim_penggemar = pd.DataFrame({'User_ID': df_clean['User_ID'].unique()})
dim_penggemar['Nama_User'] = "Anonymous Fan " + dim_penggemar['User_ID']
dim_penggemar['Fan_Status'] = np.where(np.random.rand(len(dim_penggemar)) > 0.8, 'Subscriber', 'Standard')

print("✅ 5 Tabel Dimensi berhasil dibentuk.")

✅ 5 Tabel Dimensi berhasil dibentuk.


### **Tahap 3: Membangun Tabel Fakta (Fact_Sales)**
Tabel Fakta adalah pusat dari *Star Schema*. Di sini kita menyatukan seluruh *Foreign Key* (ID) dan menghitung *Measures* (metrik kuantitatif) seperti Gross Revenue, Bandcamp Cut (10-15%), dan Net Artist Revenue.

In [4]:
# Kalkulasi Revenue Split
df_clean['Quantity'] = 1
df_clean['Gross_Rev'] = df_clean['amount_paid_usd']

# Menentukan persentase potongan (Digital 15%, Physical 10%, Bandcamp Friday 0%)
df_clean['Cut_Rate'] = np.where(df_clean['item_type'] == 'p', 0.10, 0.15)
df_clean['Cut_Rate'] = np.where(df_clean.merge(dim_waktu, on='Time_ID')['Bandcamp_Friday'], 0.0, df_clean['Cut_Rate'])

df_clean['Bandcamp_Cut'] = round(df_clean['Gross_Rev'] * df_clean['Cut_Rate'], 2)
df_clean['Artist_Revenue'] = df_clean['Gross_Rev'] - df_clean['Bandcamp_Cut']

# Membentuk Tabel Fakta
fact_sales = df_clean[['_id', 'Item_ID', 'Artist_ID', 'User_ID', 'Time_ID', 'Location_ID', 
                       'Quantity', 'Gross_Rev', 'Bandcamp_Cut', 'Artist_Revenue']]
fact_sales.rename(columns={'_id': 'Order_ID'}, inplace=True)

print("✅ Tabel Fakta (Fact_Sales) berhasil dihitung.")
display(fact_sales.head(3))

✅ Tabel Fakta (Fact_Sales) berhasil dihitung.


,Order_ID,Item_ID,Artist_ID,User_ID,Time_ID,Location_ID,Quantity,Gross_Rev,Bandcamp_Cut,Artist_Revenue
0,1599688803.5175&//girlbanddublin.bandcamp.com/...,ITM-3F13430D,ART-08BDE1AC,USR-103,20200909,LOC-7885444A,1,9.99,1.50,8.49
1,1599688805.27838&//maharettarecords.bandcamp.c...,ITM-54294AF1,ART-A1C6EC60,USR-436,20200909,LOC-75778BF8,1,1.30,0.20,1.10
2,1599688805.90646&//maharettarecords.bandcamp.c...,ITM-FE0D85E3,ART-48DFDF52,USR-349,20200909,LOC-75778BF8,1,3.90,0.58,3.32


### **Tahap 4a: Load ke SQLite**
Pada tahap ini, kita menyimpan skema Data Warehouse (5 Tabel Dimensi dan 1 Tabel Fakta) ke dalam database lokal SQLite agar data terpusat dan aman.

In [5]:
conn = sqlite3.connect('database/bandcamp_dw.db')

# Memasukkan DataFrame ke dalam tabel SQLite
dim_waktu.to_sql('Dim_Waktu', conn, if_exists='replace', index=False)
dim_lokasi.to_sql('Dim_Lokasi', conn, if_exists='replace', index=False)
dim_artis.to_sql('Dim_Artis', conn, if_exists='replace', index=False)
dim_item.to_sql('Dim_Item', conn, if_exists='replace', index=False)
dim_penggemar.to_sql('Dim_Penggemar', conn, if_exists='replace', index=False)
fact_sales.to_sql('Fact_Sales', conn, if_exists='replace', index=False)

conn.close()
print("✅ Tahap 4a Selesai: Data tersimpan di database/bandcamp_dw.db")

✅ Tahap 4a Selesai: Data tersimpan di database/bandcamp_dw.db


### **Tahap 4b: Ekspor Data Warehouse ke CSV**
Setelah data tersimpan di *database*, kita mengekstrak tabel-tabel tersebut ke dalam bentuk CSV di folder `etl/`. Format CSV ini sangat fleksibel dan mudah diimpor ke berbagai platform Business Intelligence (seperti Google Looker Studio atau Tableau).

In [6]:
conn = sqlite3.connect('database/bandcamp_dw.db')

tables = ['Fact_Sales', 'Dim_Item', 'Dim_Waktu', 'Dim_Penggemar', 'Dim_Artis', 'Dim_Lokasi']

for table in tables:
    df_export = pd.read_sql_query(f"SELECT * FROM {table}", conn)
    df_export.to_csv(f'etl/export_{table}.csv', index=False)

conn.close()
print("✅ Tahap 4b Selesai: Seluruh tabel DWH berhasil diekspor ke folder etl/")

✅ Tahap 4b Selesai: Seluruh tabel DWH berhasil diekspor ke folder etl/


### **Tahap 5: Eksekusi Advanced OLAP Queries**
Tahap terakhir adalah mengolah metrik lanjutan untuk menjawab 7 *Business Questions*. Kita akan menggunakan SQL untuk menghitung Average Transaction Value (ATV), Average Revenue Per User (ARPU), dan membandingkan rata-rata pendapatan harian agar *insight* yang dihasilkan adil dan akurat.

In [7]:
conn = sqlite3.connect('database/bandcamp_dw.db')

queries = {
    "1_revenue_bulanan": """
        SELECT w.Tahun, w.Bulan, 
               COUNT(f.Order_ID) as Total_Transaksi,
               ROUND(SUM(f.Gross_Rev), 2) as Total_Gross_Revenue, 
               ROUND(SUM(f.Artist_Revenue), 2) as Total_Artist_Revenue,
               ROUND(SUM(f.Gross_Rev) / COUNT(f.Order_ID), 2) as Avg_Transaction_Value
        FROM Fact_Sales f JOIN Dim_Waktu w ON f.Time_ID = w.Time_ID 
        GROUP BY w.Tahun, w.Bulan ORDER BY w.Tahun, w.Bulan;
    """,
    "2_revenue_genre": """
        SELECT a.Primary_Genre, a.Nama_Artis, 
               COUNT(f.Order_ID) as Item_Terjual,
               ROUND(SUM(f.Gross_Rev), 2) as Total_Revenue,
               ROUND(AVG(f.Gross_Rev), 2) as Harga_Rata_Rata
        FROM Fact_Sales f JOIN Dim_Artis a ON f.Artist_ID = a.Artist_ID 
        GROUP BY a.Primary_Genre, a.Nama_Artis ORDER BY Total_Revenue DESC;
    """,
    "3_bandcamp_friday": """
        SELECT w.Bandcamp_Friday, 
               COUNT(DISTINCT w.Tanggal) as Jumlah_Hari,
               COUNT(f.Order_ID) as Total_Transaksi, 
               ROUND(SUM(f.Gross_Rev), 2) as Total_Revenue,
               ROUND(SUM(f.Gross_Rev) / COUNT(DISTINCT w.Tanggal), 2) as Avg_Revenue_Per_Hari
        FROM Fact_Sales f JOIN Dim_Waktu w ON f.Time_ID = w.Time_ID 
        GROUP BY w.Bandcamp_Friday;
    """,
    "4_negara_fisik": """
        SELECT l.Negara, 
               COUNT(f.Order_ID) as Jumlah_Pembelian_Fisik,
               ROUND(SUM(f.Gross_Rev), 2) as Nilai_Transaksi_Fisik,
               ROUND(AVG(f.Gross_Rev), 2) as Avg_Spend_Per_Order
        FROM Fact_Sales f JOIN Dim_Lokasi l ON f.Location_ID = l.Location_ID
        JOIN Dim_Item i ON f.Item_ID = i.Item_ID WHERE i.Tipe_Item = 'Physical / Merch' 
        GROUP BY l.Negara ORDER BY Nilai_Transaksi_Fisik DESC;
    """,
    "5_top_artis_merch": """
        SELECT a.Nama_Artis, 
               COUNT(f.Order_ID) as Merch_Terjual,
               ROUND(SUM(f.Gross_Rev), 2) as Pendapatan_Kotor,
               ROUND(SUM(f.Bandcamp_Cut), 2) as Total_Potongan_Bandcamp,
               ROUND(SUM(f.Artist_Revenue), 2) as Pendapatan_Bersih_Artis
        FROM Fact_Sales f JOIN Dim_Artis a ON f.Artist_ID = a.Artist_ID
        JOIN Dim_Item i ON f.Item_ID = i.Item_ID WHERE i.Tipe_Item = 'Physical / Merch' 
        GROUP BY a.Nama_Artis ORDER BY Pendapatan_Bersih_Artis DESC LIMIT 10;
    """,
    "6_tipe_vs_negara": """
        SELECT l.Negara, w.Kuartal, i.Tipe_Item, 
               COUNT(f.Order_ID) as Volume_Penjualan,
               ROUND(SUM(f.Gross_Rev), 2) as Total_Revenue
        FROM Fact_Sales f JOIN Dim_Lokasi l ON f.Location_ID = l.Location_ID
        JOIN Dim_Waktu w ON f.Time_ID = w.Time_ID JOIN Dim_Item i ON f.Item_ID = i.Item_ID 
        GROUP BY l.Negara, w.Kuartal, i.Tipe_Item ORDER BY Total_Revenue DESC;
    """,
    "7_subscriber_vs_standard": """
        SELECT p.Fan_Status, i.Tipe_Item, 
               COUNT(DISTINCT p.User_ID) as Jumlah_Unik_Fans,
               COUNT(f.Order_ID) as Total_Transaksi,
               ROUND(SUM(f.Gross_Rev), 2) as Total_Belanja,
               ROUND(SUM(f.Gross_Rev) / COUNT(DISTINCT p.User_ID), 2) as Avg_Spend_Per_Fan
        FROM Fact_Sales f JOIN Dim_Penggemar p ON f.User_ID = p.User_ID
        JOIN Dim_Item i ON f.Item_ID = i.Item_ID 
        GROUP BY p.Fan_Status, i.Tipe_Item;
    """
}

# Mengeksekusi dan mengekspor hasil ke CSV
for filename, sql_query in queries.items():
    df_olap = pd.read_sql_query(sql_query, conn)
    df_olap.to_csv(f'olap/hasil_query{filename}.csv', index=False)

conn.close()
print("🚀 Tahap 5 Selesai: 7 Output Analisis Bisnis berhasil diekstrak ke folder olap!")

🚀 Tahap 5 Selesai: 7 Output Analisis Bisnis berhasil diekstrak ke folder olap!
